# Ski_times_poc notebook 1

This notebook reads in dummy datasets that are constructed of a baseline queue wait / 
lift time, average ski down times (assumed to be Weibull distributed) and normal
noise.  

The goal of this notebook is to be able to determine the difference in baseline queue 
wait / lift time. Because the dummy data takes this queue wait time as an input, we can
tell whether this approach is successful by comparing the detected difference in wait time
to the wait time assumed in the dataset creation.  

Outline of approach is to construct the cumulative distribution function (CDF) for each 
input dataset. Then to take the difference between each CDF to determine the average
horizontal difference between distributions.  

Goals:  
    - To test the viability of the approach.  (this notebook).  
    - To test whether mean horizontal distance is an appropriate measure (can test against max, min and median).  (this notebook).  
    - To indicate whether a separate calibration step is required to estimate queue time (i.e., does a differing levels of noise or changes in skiing behaviour between runs reduce the accuracy of the queue estimate).  (notebook 2).  


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lib.data_generation import get_dummy_returns_data


## Test viability of approach on simple data, large sample


In [ ]:
# Set input delays
SHORT_QUEUE_TIME = 15
MID_QUEUE_TIME = 45
LONG_QUEUE_TIME = 90

In [ ]:
# Only difference between datasets is queue time, keep all else the same with large sample
short_queue_data = get_dummy_returns_data(1000, SHORT_QUEUE_TIME, 30, 10, 15, random_seed=1)
mid_queue_data = get_dummy_returns_data(1000, MID_QUEUE_TIME, 30, 10, 15, random_seed=2)
long_queue_data = get_dummy_returns_data(1000, LONG_QUEUE_TIME, 30, 10, 15, random_seed=3)
print(f"Short queue data summary: \n{short_queue_data.describe()}")
print(f"Mid queue data summary: \n{mid_queue_data.describe()}")
print(f"Long queue data summary: \n{long_queue_data.describe()}")

In [ ]:
# Create histogram for ski down times
sd_fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True)
short_queue_bins = np.arange(0, short_queue_data["ski_down_time"].max(), 1)
mid_queue_bins = np.arange(0, mid_queue_data["ski_down_time"].max(), 1)
long_queue_bins = np.arange(0, long_queue_data["ski_down_time"].max(), 1)
ax1.hist(short_queue_data["ski_down_time"], bins=short_queue_bins)
ax1.set_title('short queue data')
ax1.tick_params(axis='x', labelbottom=True)
ax2.hist(mid_queue_data["ski_down_time"], bins=mid_queue_bins)
ax2.set_title('mid queue data')
ax2.tick_params(axis='x', labelbottom=True)
ax3.hist(long_queue_data["ski_down_time"], bins=long_queue_bins)
ax3.set_title('long queue data')
sd_fig.tight_layout()
plt.show()

Ski down times look reasonable. Large spike at the minimum ski-down time. This 
represents competent skiers who ski directly down the run back to the gate immediately.
Distribution of ski down times above this, quite well spread, representing slower skiers
and skiers who visit other runs / do not ski directly back to the original gate.  

NB: This is ski down time only, so no rightward shift based on queue time is as expected

In [ ]:
# Create histogram of total times including queue time, ski down time and normal noise
tot_fig, (ax4, ax5, ax6) = plt.subplots(3, 1, sharex=True)
short_queue_bins = np.arange(0, short_queue_data["total_return_time"].max(), 1)
mid_queue_bins = np.arange(0, mid_queue_data["total_return_time"].max(), 1)
long_queue_bins = np.arange(0, long_queue_data["total_return_time"].max(), 1)
ax4.hist(short_queue_data["total_return_time"], bins=short_queue_bins)
ax5.hist(mid_queue_data["total_return_time"], bins=mid_queue_bins)
ax6.hist(long_queue_data["total_return_time"], bins=long_queue_bins)
plt.show()

Datasets look reasonable. Rightward skewed data with more pronounced rightward skew for
dataset with shorter queues. Difference in queue times causes clear rightward shift of
data. This is good for testing the simple case. We should be able to pick this shift up

## Calculate quantiles and construct cumulative distribution function


In [ ]:
def get_quantiles_df(input_df: pd.DataFrame, col_name: str) -> pd.DataFrame:
    input_df = input_df.copy().sort_values(by=col_name,ascending=True)
    quantiles = np.arange(0, 101) / 100
    quantiles_df = input_df[col_name].quantile(quantiles).to_frame().reset_index(names='quantiles')
    return quantiles_df

In [ ]:
short_queue_quantiles = get_quantiles_df(short_queue_data, 'total_return_time')
mid_queue_quantiles = get_quantiles_df(mid_queue_data, 'total_return_time')
long_queue_quantiles = get_quantiles_df(long_queue_data, 'total_return_time')
plt.plot(short_queue_quantiles['total_return_time'], short_queue_quantiles['quantiles'], color='b', label='short_queue')
plt.plot(mid_queue_quantiles['total_return_time'], mid_queue_quantiles['quantiles'], color='r', label='mid_queue')
plt.plot(long_queue_quantiles['total_return_time'], long_queue_quantiles['quantiles'], color='k', label='long_queue')
plt.legend()
plt.xlabel('Return Time')
plt.ylabel('Quantile')
plt.show()

In [ ]:
# Merge into one table
quantiles = short_queue_quantiles.merge(
        mid_queue_quantiles,
        on='quantiles',
        how='inner',
        suffixes=('_short', None)
    ).merge(
        long_queue_quantiles,
        on='quantiles',
        how='inner',
        suffixes=('_mid', '_long')
    )
quantiles

# Results

In [ ]:
short_mid_delta_mean = abs((quantiles['total_return_time_short'] - quantiles['total_return_time_mid']).mean())
short_mid_delta_median = abs((quantiles['total_return_time_short'] - quantiles['total_return_time_mid']).median())
short_mid_delta_max= (quantiles['total_return_time_short'] - quantiles['total_return_time_mid']).abs().max()
short_mid_delta_min= (quantiles['total_return_time_short'] - quantiles['total_return_time_mid']).abs().min()

actual = abs(MID_QUEUE_TIME - SHORT_QUEUE_TIME)
print(f"Actual delta between short and mid queues: {actual} minutes.")
print(f"Mean delta between short and mid queues: {round(short_mid_delta_mean, 2)} minutes. Percentage error: {round(abs(actual - short_mid_delta_mean)/actual*100, 2)}%.")
print(f"Median delta between short and mid queues: {round(short_mid_delta_median, 2)} minutes. Percentage error: {round(abs(actual - short_mid_delta_median)/actual*100,2)}%.")
print(f"Max delta between short and mid queues: {round(short_mid_delta_max, 2)} minutes. Percentage error: {round(abs(actual - short_mid_delta_max)/actual*100,2)}%.")
print(f"Min delta between short and mid queues: {round(short_mid_delta_min, 2)} minutes. Percentage error: {round(abs(actual - short_mid_delta_min)/actual*100,2)}%.")

In [ ]:
mid_long_delta_mean = abs((quantiles['total_return_time_mid'] - quantiles['total_return_time_long']).mean())
mid_long_delta_median = abs((quantiles['total_return_time_mid'] - quantiles['total_return_time_long']).median())
mid_long_delta_max= (quantiles['total_return_time_mid'] - quantiles['total_return_time_long']).abs().max()
mid_long_delta_min= (quantiles['total_return_time_mid'] - quantiles['total_return_time_long']).abs().min()

actual = abs(LONG_QUEUE_TIME - MID_QUEUE_TIME)
print(f"Actual delta between mid and long queues: {actual} minutes.")
print(f"Mean delta between mid and long queues: {round(mid_long_delta_mean, 2)} minutes. Percentage error: {round(abs(actual - mid_long_delta_mean)/actual*100, 2)}%.")
print(f"Median delta between mid and long queues: {round(mid_long_delta_median, 2)} minutes. Percentage error: {round(abs(actual - mid_long_delta_median)/actual*100, 2)}%.")
print(f"Max delta between mid and long queues: {round(mid_long_delta_max, 2)} minutes. Percentage error: {round(abs(actual - mid_long_delta_max)/actual*100, 2)}%.")
print(f"Min delta between mid and long queues: {round(mid_long_delta_min, 2)} minutes. Percentage error: {round(abs(actual - mid_long_delta_min)/actual*100, 2)}%.")


In [ ]:
short_long_delta_mean = abs((quantiles['total_return_time_short'] - quantiles['total_return_time_long']).mean())
short_long_delta_median = abs((quantiles['total_return_time_short'] - quantiles['total_return_time_long']).median())
short_long_delta_max= (quantiles['total_return_time_short'] - quantiles['total_return_time_long']).abs().max()
short_long_delta_min= (quantiles['total_return_time_short'] - quantiles['total_return_time_long']).abs().min()

actual = abs(LONG_QUEUE_TIME - SHORT_QUEUE_TIME)
print(f"Actual delta between short and long queues: {actual} minutes.")
print(f"Mean delta between short and long queues: {round(short_long_delta_mean, 2)} minutes. Percentage error: {round(abs(actual - short_long_delta_mean)/actual*100, 2)}%.")
print(f"Median delta between short and long queues: {round(short_long_delta_median, 2)} minutes. Percentage error: {round(abs(actual - short_long_delta_median)/actual*100,2)}%.")
print(f"Max delta between short and long queues: {round(short_long_delta_max, 2)} minutes. Percentage error: {round(abs(actual - short_long_delta_max)/actual*100,2)}%.")
print(f"Min delta between short and long queues: {round(short_long_delta_min, 2)} minutes. Percentage error: {round(abs(actual - short_long_delta_min)/actual*100,2)}%.")


# Conclusion  

OK, so the approach works, it's consistently able to estimate the difference in queue
times between short and long queues when given reasonable best case scenario data (i.e.,
large sample size (1000), consistent normally distributed noise, no competing effects).

Both mean and median differences provide similarly performant methods of calculating the
horizontal distance between cumulative distribution functions. Max and Min are discounted for being less reliable than mean and median on the simple data.